# ESP-Fi HAR Table 4 Benchmark (Colab)

This notebook benchmarks all deep models from `ModelCode/run.py` and adds traditional ML baselines (`SVM`, `RF`, `LR`) to reproduce Table 4 from the paper.

What this notebook implements:
- 8-fold leave-one-subject-out (LOSO) evaluation per scene (when full scene data is present)
- Deep models: `LSTM`, `GRU`, `CNN`, `ResNet18`, `EfficientNet-Lite`, `Transformer`, `MobileNetV3`
- Traditional models: `SVM`, `RandomForest`, `LogisticRegression`
- Metrics: Accuracy mean/std and macro-F1 per scene

Important notes:
- The paper explicitly states LOSO + identical splits/settings + AdamW for deep models.
- The paper text does **not** provide explicit classical-model hyperparameters; defaults are exposed below so you can tune them.
- If your extracted dataset only has one scene (for example scene 3), this notebook will run on available scenes and report partial reproduction.

In [ ]:
# Colab dependency setup (safe to rerun)
!pip -q install scipy scikit-learn pandas tqdm einops

In [ ]:
import os
import re
import sys
import copy
import random
import subprocess
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from tqdm.auto import tqdm

In [ ]:
# Resolve repository root and import model definitions

def resolve_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/ESP-Fi-HAR')]
    for c in candidates:
        if (c / 'ModelCode' / 'ESP_Fi_model.py').exists():
            return c

    clone_target = Path('/content/ESP-Fi-HAR')
    if not clone_target.exists():
        print('Cloning ESP-Fi-HAR repository...')
        subprocess.run(
            ['git', 'clone', 'https://github.com/AutoSmartGroup/ESP-Fi-HAR.git', str(clone_target)],
            check=True,
        )
    return clone_target

REPO_ROOT = resolve_repo_root()
MODELCODE_DIR = REPO_ROOT / 'ModelCode'
if str(MODELCODE_DIR) not in sys.path:
    sys.path.insert(0, str(MODELCODE_DIR))

print('REPO_ROOT =', REPO_ROOT)
print('MODELCODE_DIR =', MODELCODE_DIR)

In [ ]:
from ESP_Fi_model import (
    CNN,
    ESP_Fi_ResNet18,
    ESP_Fi_GRU,
    ESP_Fi_LSTM,
    ESP_Fi_Transformer,
    MobileNetV3,
    EfficientNetLite,
)

ACTIVITY_ORDER = ['arm_wave', 'fall', 'jump', 'run', 'squat', 'turn', 'walk']
SCENE_NAME = {1: 'Corridor', 2: 'Office', 3: 'Meeting room', 4: 'Laboratory'}

# Fallback mapping from activity-id (Z in X-Y-Z-M) to activity name.
# This mapping is inferred from released split files; if your dataset embeds
# clear folder names, folder names are preferred.
DEFAULT_ACTIVITY_ID_TO_NAME = {
    1: 'run',
    2: 'walk',
    3: 'jump',
    4: 'squat',
    5: 'arm_wave',
    6: 'turn',
    7: 'fall',
}

DATA_ROOT = Path(os.environ.get('ESP_FI_DATA_ROOT', MODELCODE_DIR / 'Data'))
print('DATA_ROOT =', DATA_ROOT)

In [ ]:
MAT_NAME_RE = re.compile(r'^(\d+)-(\d+)-(\d+)-(\d+)$')


def build_manifest(data_root: Path) -> tuple[pd.DataFrame, dict[int, str]]:
    mat_files = sorted(data_root.rglob('*.mat'))
    if not mat_files:
        raise FileNotFoundError(f'No .mat files found under: {data_root}')

    # Infer activity-id -> activity-name from folder names when possible.
    inferred = {}
    rows = []

    for p in mat_files:
        m = MAT_NAME_RE.match(p.stem)
        if not m:
            continue

        scene, subject, activity_id, trial = map(int, m.groups())
        parent_name = p.parent.name.strip().lower().replace(' ', '_')

        if parent_name in ACTIVITY_ORDER:
            if activity_id in inferred and inferred[activity_id] != parent_name:
                raise ValueError(
                    f'Conflicting mapping for activity_id={activity_id}: '
                    f'{inferred[activity_id]} vs {parent_name}'
                )
            inferred[activity_id] = parent_name

        rows.append(
            {
                'path': str(p),
                'scene': scene,
                'subject': subject,
                'activity_id': activity_id,
                'trial': trial,
                'parent_activity': parent_name,
            }
        )

    if not rows:
        raise RuntimeError('No valid X-Y-Z-M .mat files were parsed.')

    activity_id_to_name = dict(DEFAULT_ACTIVITY_ID_TO_NAME)
    activity_id_to_name.update(inferred)

    clean_rows = []
    for r in rows:
        if r['parent_activity'] in ACTIVITY_ORDER:
            activity_name = r['parent_activity']
        else:
            activity_name = activity_id_to_name.get(r['activity_id'])

        if activity_name not in ACTIVITY_ORDER:
            continue

        label = ACTIVITY_ORDER.index(activity_name)
        clean_rows.append(
            {
                **r,
                'activity_name': activity_name,
                'label': label,
                'scene_name': SCENE_NAME.get(r['scene'], f'Scene {r["scene"]}'),
            }
        )

    manifest = pd.DataFrame(clean_rows).sort_values(
        ['scene', 'subject', 'activity_name', 'trial', 'path']
    ).reset_index(drop=True)

    if manifest.empty:
        raise RuntimeError('Manifest is empty after parsing labels.')

    return manifest, activity_id_to_name


def summarize_manifest(manifest: pd.DataFrame) -> None:
    print('Total samples:', len(manifest))
    print('Scenes:', sorted(manifest['scene'].unique().tolist()))
    print('Subjects:', sorted(manifest['subject'].unique().tolist()))
    print('Activities:', sorted(manifest['activity_name'].unique().tolist()))
    print('\nSamples per scene:')
    display(manifest.groupby(['scene', 'scene_name']).size().rename('n').reset_index())

In [ ]:
def make_loso_folds(
    manifest: pd.DataFrame,
    scenes_to_run: list[int] | None = None,
    max_subject_folds: int | None = None,
) -> list[dict]:
    if scenes_to_run is None:
        scenes_to_run = sorted(manifest['scene'].unique().tolist())

    folds = []
    for scene in scenes_to_run:
        scene_df = manifest[manifest['scene'] == scene]
        if scene_df.empty:
            print(f'[warn] scene={scene} missing in manifest, skipping.')
            continue

        subjects = sorted(scene_df['subject'].unique().tolist())
        if max_subject_folds is not None:
            subjects = subjects[:max_subject_folds]

        for held_out_subject in subjects:
            train_df = scene_df[scene_df['subject'] != held_out_subject]
            test_df = scene_df[scene_df['subject'] == held_out_subject]
            if train_df.empty or test_df.empty:
                continue

            folds.append(
                {
                    'scene': scene,
                    'scene_name': SCENE_NAME.get(scene, f'Scene {scene}'),
                    'held_out_subject': held_out_subject,
                    'train': train_df,
                    'test': test_df,
                }
            )

    return folds

In [ ]:
@lru_cache(maxsize=None)
def load_csi_sample(path: str) -> np.ndarray:
    mat = sio.loadmat(path)

    key_candidates = ['CSIamp', 'csi_amp', 'CSI_amp', 'amp']
    data = None
    for k in key_candidates:
        if k in mat:
            data = mat[k]
            break

    if data is None:
        raise KeyError(f'No CSI amplitude key found in {path}. Tried: {key_candidates}')

    x = np.asarray(data, dtype=np.float32)

    if x.shape == (950, 52):
        pass
    elif x.shape == (52, 950):
        x = x.T
    else:
        raise ValueError(f'Unexpected sample shape {x.shape} in {path}')

    # Per-sample z-score normalization (same as dataset.py)
    x = (x - x.mean()) / (x.std() + 1e-8)
    return x.reshape(1, 950, 52).astype(np.float32)


@lru_cache(maxsize=None)
def load_flat_feature(path: str) -> np.ndarray:
    return load_csi_sample(path).reshape(-1).astype(np.float32)


class ESPFiDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        x = load_csi_sample(row['path']).copy()
        y = int(row['label'])
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

In [ ]:
DEEP_MODEL_BUILDERS = {
    'CNN': lambda num_classes: CNN(num_classes),
    'ResNet18': lambda num_classes: ESP_Fi_ResNet18(num_classes),
    'GRU': lambda num_classes: ESP_Fi_GRU(num_classes),
    'LSTM': lambda num_classes: ESP_Fi_LSTM(num_classes),
    'Transformer': lambda num_classes: ESP_Fi_Transformer(num_classes),
    'MobileNetV3': lambda num_classes: MobileNetV3(num_classes),
    'EfficientNetLite': lambda num_classes: EfficientNetLite(num_classes),
}

# Deep-model training settings aligned with ModelCode/run.py + README guidance.
DEEP_HPARAMS = {
    'CNN': {'epochs': 50, 'batch_size': 32, 'lr': 1e-3, 'weight_decay': 1e-4},
    'ResNet18': {'epochs': 50, 'batch_size': 32, 'lr': 1e-3, 'weight_decay': 1e-4},
    'GRU': {'epochs': 100, 'batch_size': 64, 'lr': 1e-3, 'weight_decay': 1e-4},
    'LSTM': {'epochs': 100, 'batch_size': 32, 'lr': 1e-3, 'weight_decay': 1e-4},
    'Transformer': {'epochs': 100, 'batch_size': 4, 'lr': 1e-3, 'weight_decay': 1e-4},
    'MobileNetV3': {'epochs': 50, 'batch_size': 32, 'lr': 1e-3, 'weight_decay': 1e-4},
    'EfficientNetLite': {'epochs': 50, 'batch_size': 32, 'lr': 1e-3, 'weight_decay': 1e-4},
}

In [ ]:
def set_seed(seed: int = 666):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def evaluate_deep(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[float, float]:
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            p = torch.argmax(logits, dim=1)
            preds.extend(p.cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return float(acc), float(f1)


def train_eval_deep_fold(
    model_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    device: torch.device,
    seed: int = 666,
    verbose: bool = False,
) -> dict:
    set_seed(seed)

    hp = DEEP_HPARAMS[model_name]
    model = DEEP_MODEL_BUILDERS[model_name](num_classes=len(ACTIVITY_ORDER)).to(device)

    train_loader = DataLoader(
        ESPFiDataset(train_df),
        batch_size=hp['batch_size'],
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        ESPFiDataset(test_df),
        batch_size=hp['batch_size'],
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hp['lr'],
        weight_decay=hp['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hp['epochs'])

    best_train_acc = -1.0
    best_state = None

    for epoch in range(hp['epochs']):
        model.train()
        correct = 0
        total = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            pred = torch.argmax(logits, dim=1)
            correct += int((pred == y).sum().item())
            total += int(y.numel())

        train_acc = correct / max(total, 1)
        if train_acc > best_train_acc:
            best_train_acc = train_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f'[{model_name}] epoch {epoch+1}/{hp["epochs"]} train_acc={train_acc:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)

    acc, f1 = evaluate_deep(model, test_loader, device)
    return {
        'acc': acc,
        'f1': f1,
        'best_train_acc': best_train_acc,
        'epochs': hp['epochs'],
        'batch_size': hp['batch_size'],
        'lr': hp['lr'],
        'weight_decay': hp['weight_decay'],
    }

In [ ]:
ML_HPARAMS = {
    # Paper does not list exact classical-model hyperparameters.
    # These are explicit defaults you can tune.
    'SVM': {'C': 10.0, 'kernel': 'rbf', 'gamma': 'scale'},
    'RF': {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1},
    'LR': {'C': 1.0, 'solver': 'saga', 'max_iter': 600},
}


def build_ml_model(model_name: str, seed: int = 666):
    if model_name == 'SVM':
        hp = ML_HPARAMS['SVM']
        return Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(C=hp['C'], kernel=hp['kernel'], gamma=hp['gamma'])),
        ])

    if model_name == 'RF':
        hp = ML_HPARAMS['RF']
        return RandomForestClassifier(
            n_estimators=hp['n_estimators'],
            max_depth=hp['max_depth'],
            min_samples_leaf=hp['min_samples_leaf'],
            random_state=seed,
            n_jobs=-1,
        )

    if model_name == 'LR':
        hp = ML_HPARAMS['LR']
        return Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                C=hp['C'],
                solver=hp['solver'],
                max_iter=hp['max_iter'],
                multi_class='multinomial',
                random_state=seed,
                n_jobs=-1,
            )),
        ])

    raise ValueError(f'Unsupported ML model: {model_name}')


def train_eval_ml_fold(
    model_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    seed: int = 666,
) -> dict:
    set_seed(seed)

    X_train = np.stack([load_flat_feature(p) for p in train_df['path'].tolist()], axis=0)
    y_train = train_df['label'].to_numpy(dtype=np.int64)

    X_test = np.stack([load_flat_feature(p) for p in test_df['path'].tolist()], axis=0)
    y_test = test_df['label'].to_numpy(dtype=np.int64)

    model = build_ml_model(model_name, seed=seed)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='macro')

    return {'acc': float(acc), 'f1': float(f1)}

In [ ]:
# ==============================
# User configuration
# ==============================
SCENES_TO_RUN = [1, 2, 3, 4]   # any subset of [1,2,3,4]
MAX_SUBJECT_FOLDS = None       # set small int for quick debug runs

RUN_CLASSICAL = True
RUN_DEEP = True

ML_MODELS_TO_RUN = ['SVM', 'RF', 'LR']
DEEP_MODELS_TO_RUN = ['LSTM', 'GRU', 'CNN', 'ResNet18', 'EfficientNetLite', 'Transformer', 'MobileNetV3']

SEED = 666
VERBOSE_EPOCHS = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

manifest, inferred_map = build_manifest(DATA_ROOT)
print('Inferred activity-id mapping:', inferred_map)
summarize_manifest(manifest)

folds = make_loso_folds(
    manifest,
    scenes_to_run=SCENES_TO_RUN,
    max_subject_folds=MAX_SUBJECT_FOLDS,
)
print('Total LOSO folds:', len(folds))
if len(folds) == 0:
    raise RuntimeError('No valid folds were generated. Check DATA_ROOT and SCENES_TO_RUN.')

In [ ]:
results = []

# Classical ML models
if RUN_CLASSICAL:
    for model_name in ML_MODELS_TO_RUN:
        for fold in tqdm(folds, desc=f'ML {model_name}'):
            out = train_eval_ml_fold(model_name, fold['train'], fold['test'], seed=SEED)
            results.append(
                {
                    'model_type': 'ML',
                    'model': model_name,
                    'scene': fold['scene'],
                    'scene_name': fold['scene_name'],
                    'held_out_subject': fold['held_out_subject'],
                    **out,
                }
            )

# Deep models from run.py
if RUN_DEEP:
    for model_name in DEEP_MODELS_TO_RUN:
        for fold in tqdm(folds, desc=f'DL {model_name}'):
            out = train_eval_deep_fold(
                model_name,
                fold['train'],
                fold['test'],
                device=DEVICE,
                seed=SEED,
                verbose=VERBOSE_EPOCHS,
            )
            results.append(
                {
                    'model_type': 'DL',
                    'model': model_name,
                    'scene': fold['scene'],
                    'scene_name': fold['scene_name'],
                    'held_out_subject': fold['held_out_subject'],
                    **out,
                }
            )

results_df = pd.DataFrame(results)
print('Completed rows:', len(results_df))
display(results_df.head())

In [ ]:
if results_df.empty:
    raise RuntimeError('No results were produced.')

summary = (
    results_df
    .groupby(['model_type', 'model', 'scene', 'scene_name'], as_index=False)
    .agg(
        folds=('held_out_subject', 'nunique'),
        acc_mean=('acc', 'mean'),
        acc_std=('acc', 'std'),
        f1_mean=('f1', 'mean'),
    )
)

for c in ['acc_mean', 'acc_std', 'f1_mean']:
    summary[c] = summary[c] * 100.0

summary = summary.sort_values(['model_type', 'model', 'scene']).reset_index(drop=True)
display(summary.round(2))

In [ ]:
# Table 4 values from the paper (percent)
paper_rows = [
    ('SVM', 1, 24.64, 8.01, 19.00), ('SVM', 2, 24.11, 7.09, 19.08), ('SVM', 3, 20.00, 9.20, 15.83), ('SVM', 4, 19.82, 10.61, 14.31),
    ('RF', 1, 23.57, 7.14, 19.01), ('RF', 2, 21.25, 6.16, 17.19), ('RF', 3, 24.64, 12.37, 21.82), ('RF', 4, 17.68, 7.59, 13.47),
    ('LR', 1, 19.29, 7.63, 16.87), ('LR', 2, 22.86, 3.98, 19.51), ('LR', 3, 16.43, 5.05, 12.66), ('LR', 4, 14.82, 2.36, 11.14),
    ('LSTM', 1, 32.15, 13.40, 29.82), ('LSTM', 2, 38.02, 14.21, 33.80), ('LSTM', 3, 36.20, 12.04, 33.84), ('LSTM', 4, 32.73, 13.25, 30.21),
    ('GRU', 1, 30.93, 3.78, 28.74), ('GRU', 2, 38.02, 8.63, 33.00), ('GRU', 3, 34.55, 4.21, 31.17), ('GRU', 4, 28.38, 8.98, 24.56),
    ('ResNet18', 1, 63.59, 14.91, 60.81), ('ResNet18', 2, 70.11, 19.60, 64.00), ('ResNet18', 3, 65.53, 18.22, 61.26), ('ResNet18', 4, 58.33, 22.73, 49.14),
    ('EfficientNetLite', 1, 61.60, 10.60, 57.13), ('EfficientNetLite', 2, 63.74, 8.74, 58.80), ('EfficientNetLite', 3, 61.98, 5.14, 57.93), ('EfficientNetLite', 4, 50.35, 6.96, 46.27),
    ('CNN', 1, 57.55, 11.43, 53.20), ('CNN', 2, 48.31, 8.02, 43.80), ('CNN', 3, 48.26, 11.14, 42.70), ('CNN', 4, 52.61, 12.19, 46.89),
    ('Transformer', 1, 54.43, 12.30, 49.85), ('Transformer', 2, 41.93, 9.47, 38.20), ('Transformer', 3, 47.51, 10.71, 42.43), ('Transformer', 4, 50.33, 15.11, 46.35),
    ('MobileNetV3', 1, 53.64, 13.24, 47.28), ('MobileNetV3', 2, 51.82, 11.85, 47.00), ('MobileNetV3', 3, 53.28, 15.18, 46.80), ('MobileNetV3', 4, 43.31, 16.25, 38.43),
]

paper_df = pd.DataFrame(paper_rows, columns=['model', 'scene', 'paper_acc_mean', 'paper_acc_std', 'paper_f1_mean'])
paper_df['scene_name'] = paper_df['scene'].map(SCENE_NAME)

comparison = summary.merge(paper_df, on=['model', 'scene', 'scene_name'], how='inner')
comparison['delta_acc_mean'] = comparison['acc_mean'] - comparison['paper_acc_mean']
comparison['delta_acc_std'] = comparison['acc_std'] - comparison['paper_acc_std']
comparison['delta_f1_mean'] = comparison['f1_mean'] - comparison['paper_f1_mean']

display(comparison.sort_values(['model_type', 'model', 'scene']).round(2))

In [ ]:
OUT_DIR = REPO_ROOT / 'notebooks' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

raw_path = OUT_DIR / 'benchmark_fold_results.csv'
summary_path = OUT_DIR / 'benchmark_scene_summary.csv'
comparison_path = OUT_DIR / 'benchmark_vs_paper_table4.csv'

results_df.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)

if 'comparison' in globals() and not comparison.empty:
    comparison.to_csv(comparison_path, index=False)

print('Saved:', raw_path)
print('Saved:', summary_path)
if comparison_path.exists():
    print('Saved:', comparison_path)